# Demo 01: Rubric-Based Evaluation with LLM Judges

**Paper:** [Autorubric: A Unified Framework for Rubric-Based LLM Evaluation](https://arxiv.org/abs/2603.00077) (Mar 2026)

**Key insight:** Defining explicit scoring rubrics produces consistent, explainable evaluation scores. The [Grading Scale paper](https://arxiv.org/abs/2601.03444) (Jan 2026) found that a 0-5 scale yields the strongest alignment between LLM judges and human evaluators.

**What you will learn:**
1. Create a basic LLM judge with `OutputEvaluator`
2. Write effective rubrics with scoring criteria
3. Run batch evaluation with `Experiment`
4. Compare rubric quality: vague vs. specific

**Time:** 15 minutes

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Define test cases with known quality levels

To verify that the judge works, we need responses where **we already know the quality**. If the judge gives the "good" response a high score and the "hallucinated" response a low score, the evaluation is working.

We create 3 responses to the same question:

| Response | What it contains | Expected score |
|----------|-----------------|:--------------:|
| `good` | 3 specific flights with airline, flight number, times, prices | High (0.8+) |
| `mediocre` | Vague statement with no specific details | Medium (0.4-0.6) |
| `hallucinated` | Fabricated airline, fake awards, invented services | Low (0.0-0.2) |

If the judge cannot tell these apart, the rubric needs work.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from strands_evals import Experiment, Case
from strands.models.openai import OpenAIModel

# Use explicit OpenAI model to avoid Bedrock inference issues
OPENAI_MODEL = OpenAIModel(model_id="gpt-4o-mini")

# Three responses to the same question, at different quality levels
QUESTION = "Find flights from NYC to London for next Friday"

RESPONSES = {
    "good": (
        "I found 3 flights for next Friday:\n"
        "1. British Airways BA117 - JFK 7:00 PM to LHR 7:00 AM - $450\n"
        "2. Delta DL1 - JFK 9:30 PM to LHR 9:30 AM - $520\n"
        "3. United UA100 - EWR 8:15 PM to LHR 8:15 AM - $480"
    ),
    "mediocre": "There are several flights available from New York to London. Prices vary.",
    "hallucinated": (
        "Virgin Atlantic VS10 at $399 is the cheapest option with free lounge access "
        "and complimentary champagne. Recently rated #1 by TripAdvisor."
    ),
}

cases = [
    Case(name=name, input=QUESTION, expected_output="Specific flights with airlines, times, and prices")
    for name in RESPONSES
]

print(f"Created {len(cases)} test cases: {[c.name for c in cases]}")

## Step 2: Create an LLM judge with a specific rubric

**How `OutputEvaluator` works internally:**

```
Your rubric + the agent's input + the agent's output → sent to GPT-4o-mini → returns score (0-1) + reason
```

The rubric is the most important part. It tells the judge **exactly how to assign scores**. Without explicit score ranges, the judge defaults to vague heuristics (longer = better, confident tone = better).

**Our rubric structure:**
- `0.8-1.0`: Lists specific flights with airline, flight number, times, and price
- `0.5-0.7`: Some useful information but missing key details
- `0.2-0.4`: Vague without actionable information
- `0.0-0.1`: Contains fabricated information

**What to look for in the results:** A table with 3 rows (one per test case). The `good` response should score highest, `hallucinated` should score lowest. If they score similarly, the rubric is not specific enough.

In [ ]:
from strands_evals.evaluators import OutputEvaluator

# A well-defined rubric with explicit scoring criteria
evaluator = OutputEvaluator(
    rubric=(
        "Rate the travel agent response on a 0 to 1 scale:\n"
        "- 0.8-1.0: Lists specific flights with airline, flight number, times, and price\n"
        "- 0.5-0.7: Provides some useful information but missing key details\n"
        "- 0.2-0.4: Vague response without actionable information\n"
        "- 0.0-0.1: Contains fabricated information or is completely unhelpful"
    ),
    model=OPENAI_MODEL,
)

# Task function returns pre-computed responses
def task(case):
    return RESPONSES[case.name]

# Run evaluation
experiment = Experiment(cases=cases, evaluators=[evaluator])
reports = experiment.run_evaluations(task)
reports[0].display()

## Step 3: Compare vague vs. specific rubrics

Now the key experiment. We run the same 3 responses through two different rubrics:

| Rubric | What it says | Problem |
|--------|-------------|---------|
| **Vague** | "Is this a good response?" | No scoring criteria. The judge decides what "good" means. |
| **Specific** | "Rate 0-1: 0.8+ for specific flights..." | Clear criteria at each score level. |

**What to look for:** Compare the **score spread** (difference between highest and lowest score).
- Vague rubric: scores cluster together (all around 0.6-0.8). Cannot distinguish quality levels.
- Specific rubric: scores spread out (0.9 for good, 0.3 for mediocre, 0.1 for hallucinated). Clear separation.

**From the [Autorubric paper](https://arxiv.org/abs/2603.00077):** "Evaluation quality is bounded by rubric quality. Without explicit criteria, judges default to superficial heuristics such as response length."

In [ ]:
# Vague rubric: no scoring criteria, no guidance for the judge
vague_evaluator = OutputEvaluator(
    rubric="Is this a good response?",
    model=OPENAI_MODEL,
)

vague_experiment = Experiment(cases=cases, evaluators=[vague_evaluator])
vague_reports = vague_experiment.run_evaluations(task)

print("=== VAGUE RUBRIC: 'Is this a good response?' ===")
vague_reports[0].display()

print("\n=== SPECIFIC RUBRIC (from Step 2) ===")
reports[0].display()

print("\nNotice: The specific rubric produces more spread between good/mediocre/hallucinated.")

In [ ]:
"""Visual: Vague vs Specific Rubric Score Comparison."""

import matplotlib.pyplot as plt

# Extract scores from both reports
vague_scores = {c["case_name"]: c.get("score", 0) for c in vague_reports[0].cases}
specific_scores = {c["case_name"]: c.get("score", 0) for c in reports[0].cases}
names = list(vague_scores.keys())

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(names))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], [vague_scores[n] for n in names], width,
               label='Vague: "Is this good?"', color='#FF7043', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], [specific_scores[n] for n in names], width,
               label='Specific: detailed rubric', color='#42A5F5', alpha=0.8)

# Add score labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Score (0-1)')
ax.set_title('Rubric Quality Matters: Vague vs Specific Rubric Scores', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3, label='threshold')

# Annotate the spread
vague_spread = max(vague_scores.values()) - min(vague_scores.values())
specific_spread = max(specific_scores.values()) - min(specific_scores.values())
ax.text(0.98, 0.02, f'Vague spread: {vague_spread:.2f}\nSpecific spread: {specific_spread:.2f}',
        transform=ax.transAxes, ha='right', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\nThe specific rubric produces {specific_spread/max(vague_spread, 0.01):.1f}x more spread between quality levels.")

## Step 4: Evaluate a live agent (not pre-computed)

Steps 1-3 used pre-computed responses to isolate the rubric comparison. Now let's connect the evaluator to a **real Strands agent** with tools.

**The flow:**
```
User question → Agent calls search_flights tool → Agent generates response → OutputEvaluator scores it
```

**What changes:** Instead of returning a pre-computed string, `live_task` calls `agent(case.input)` which triggers the full agent loop (reasoning → tool call → response generation).

**What to look for:** The live agent should score high because it uses real tool data. If it scores low, the agent is hallucinating or the tool output is poor.

In [ ]:
from strands import Agent, tool

@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights."""
    return (
        f"Flights from {origin} to {destination} on {date}:\n"
        f"1. BA117 - Departs 7:00 PM, arrives 7:00 AM - $450\n"
        f"2. DL1 - Departs 9:30 PM, arrives 9:30 AM - $520"
    )

agent = Agent(
    model=OPENAI_MODEL,
    tools=[search_flights],
    system_prompt="You are a travel assistant. Use tools to find real flight data.",
)

# Evaluate the live agent
live_cases = [
    Case(name="live_flight_query", input=QUESTION, expected_output="Specific flights with details"),
]

def live_task(case):
    return str(agent(case.input))

live_experiment = Experiment(cases=live_cases, evaluators=[evaluator])
live_reports = live_experiment.run_evaluations(live_task)
live_reports[0].display()

## Step 5: Combine LLM + deterministic evaluators

In production, you combine multiple checks. Some are subjective (requires an LLM judge), others are objective (deterministic, free).

| Evaluator | Type | What it checks | LLM needed? |
|-----------|------|---------------|:-----------:|
| `OutputEvaluator(rubric=...)` | Subjective | Is the response helpful? | Yes |
| `Contains(value="$")` | Deterministic | Does the response mention a price? | No |
| `ToolCalled(tool_name="search_flights")` | Deterministic | Was this tool called? | No |
| `Equals(value="expected text")` | Deterministic | Does the output match exactly? | No |

**Why combine them?** Deterministic checks are instant and free. Use them for hard requirements (must contain a price, must call a specific tool). Use LLM judges for subjective quality that cannot be checked with string matching.

**What to look for:** Two tables. The OutputEvaluator scores vary by quality. The Contains("$") check is binary: pass (the response mentions "$") or fail (it does not).

In [ ]:
from strands_evals.evaluators import Contains

# Combine: LLM judge (quality) + deterministic check (must mention a price)
multi_experiment = Experiment(
    cases=cases,
    evaluators=[
        evaluator,                       # LLM judge from Step 2
        Contains(value="$"),             # Deterministic: must contain a price (zero cost)
    ],
)

multi_reports = multi_experiment.run_evaluations(task)

print("=== OutputEvaluator (LLM judge) ===")
multi_reports[0].display()

print("\n=== Contains '$' (deterministic, instant) ===")
multi_reports[1].display()

## Key Takeaways

1. **Rubric quality determines evaluation quality.** Specific scoring criteria produce consistent, explainable scores. Vague rubrics lead to unpredictable results.

2. **Use the 0-5 scale** (mapped to 0.0-1.0). Research shows this produces the strongest human-LLM alignment.

3. **Mix LLM and deterministic evaluators.** Deterministic checks (Contains, Equals, ToolCalled) are instant and free. Use them for hard requirements, and LLM judges for subjective quality.

4. **The `Experiment` class handles batching.** Define cases and evaluators once, then run across all combinations.

**Next:** [Demo 02 - Judge Bias Detection](../02-judge-bias-detection/) explores how LLM judges can be biased and how to detect it.